# DS · 3 · Train + register readmission model

> Verified against MS Learn 2026-09-15.

In [ ]:
import mlflow
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

mlflow.set_experiment('contoso_readmission_risk')
mlflow.autolog()

df = spark.table('features_readmission').toPandas()
X = df.drop(columns=['member_id','readmit_30d'])
y = df['readmit_30d']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

with mlflow.start_run() as run:
    model = GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    mlflow.log_metric('auc', auc)

mlflow.register_model(f'runs:/{run.info.run_id}/model', 'contoso_readmission_risk_v1')